# KLTN — V1.2: Train với 56 đặc trưng hình học + Split-by-Clip

**Mục tiêu:** Cải thiện V1.1 (F1=0.5526) bằng cách dùng `processed_data1/` (56 geometric features) của Hảo thay vì 34 toạ độ thô.

**Lý do:** Theo bài báo Hảo cite, các đặc trưng hình học (relative keypoint + bone vector + hand-hip distance) mã hoá cấu trúc cơ thể tốt hơn toạ độ thô và mô hình có khả năng học hành vi tay đưa vào túi.

**Dự kiến:** F1 V1.2 ≈ 0.65-0.75 (nếu geometric features có giá trị) hoặc vẫn ≈ 0.55 (nếu vấn đề là kiến trúc hoặc dataset).

## Trước khi chạy
1. Folder `Human-Reco/processed_data1/` phải có trên Drive (76 file `normal/` + 79 file `shoplifting/`).
2. Notebook này KHÔNG dùng `X_data.npy` (vì là 34 feat). Sẽ regenerate từ CSV.
3. Vẫn dùng `clip_ids` để split — đảm bảo so sánh fair với V1.1.

## 1. Mount Drive + cấu hình

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
ROOT_DIR = "/content/drive/MyDrive/KLTN/Human-Reco"  # ★ Sửa nếu khác
assert os.path.exists(f"{ROOT_DIR}/processed_data1"), "Thiếu processed_data1/"
OUT_DIR = f"{ROOT_DIR}/training_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"ROOT_DIR = {ROOT_DIR}")

## 2. Cài deps + import

In [ ]:
!pip install -q tensorflow scikit-learn pandas seaborn matplotlib numpy --upgrade

In [ ]:
import os, json, time, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (LSTM, Conv2D, MaxPooling2D, GlobalAveragePooling2D,
                                     BatchNormalization, Dense, Reshape, Input, Dropout, Bidirectional)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, f1_score
print("TF:", tf.__version__, "GPU:", tf.config.list_physical_devices('GPU'))
SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

## 3. Regenerate X 56-feat + clip_ids từ `processed_data1/`

Replay đúng logic `prepare_data(1).py` của Hảo với `seq_length=90, step=30`.

In [ ]:
def regenerate_56feat(processed_root, seq_length=90, step=30):
    X, y, clip_ids = [], [], []
    for label_name in ['normal', 'shoplifting']:
        label_val = 0 if label_name == 'normal' else 1
        folder = os.path.join(processed_root, label_name)
        files = sorted(glob.glob(os.path.join(folder, "*.csv")))
        for fp in files:
            clip_id = f"{label_name}/{os.path.basename(fp)}"
            data = pd.read_csv(fp).values
            if data.shape[1] != 56:
                print(f"⚠ {fp} có {data.shape[1]} cột, bỏ qua")
                continue
            for i in range(0, len(data) - seq_length + 1, step):
                X.append(data[i : i + seq_length])
                y.append(label_val)
                clip_ids.append(clip_id)
    return (np.array(X, dtype='float32'),
            np.array(y, dtype='int'),
            np.array(clip_ids))

X, y, clip_ids = regenerate_56feat(f"{ROOT_DIR}/processed_data1")
print(f"X.shape = {X.shape}")
print(f"y.shape = {y.shape}")
print(f"Số clip: {len(np.unique(clip_ids))}")
u, c = np.unique(y, return_counts=True)
for ui, ci in zip(u, c):
    print(f"  class {ui}: {ci} ({100*ci/len(y):.1f}%)")
print(f"X min/max: {X.min():.4f} / {X.max():.4f}")
print(f"X mean/std: {X.mean():.4f} / {X.std():.4f}")

## 4. Build mô hình LSTM-CNN (input shape (90, 56) thay vì (90, 34))

In [ ]:
def build_hybrid_model_56(input_shape=(90, 56), num_classes=2):
    model = Sequential([
        Input(shape=input_shape),
        LSTM(32, return_sequences=True),
        LSTM(32, return_sequences=True),
        Reshape((90, 32, 1)),
        Conv2D(64, kernel_size=(5, 5), strides=(2, 2), padding='same', activation='relu'),
        MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
        Conv2D(128, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='relu'),
        GlobalAveragePooling2D(),
        BatchNormalization(),
        Dense(num_classes, activation='softmax'),
    ])
    model.compile(optimizer=Adam(learning_rate=1e-4),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

build_hybrid_model_56().summary()

## 5. Split-by-clip 70/15/15 (giống V1.1)

In [ ]:
gss1 = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=SEED)
idx_tr, idx_rem = next(gss1.split(X, y, groups=clip_ids))
gss2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEED+1)
idx_v_loc, idx_t_loc = next(gss2.split(X[idx_rem], y[idx_rem], groups=clip_ids[idx_rem]))
idx_v = idx_rem[idx_v_loc]; idx_t = idx_rem[idx_t_loc]

X_tr, y_tr = X[idx_tr], y[idx_tr]
X_v, y_v = X[idx_v], y[idx_v]
X_t, y_t = X[idx_t], y[idx_t]

# Assert không leak
clips_tr, clips_v, clips_t = set(clip_ids[idx_tr]), set(clip_ids[idx_v]), set(clip_ids[idx_t])
assert not (clips_tr & clips_v), "LEAK train↔val"
assert not (clips_tr & clips_t), "LEAK train↔test"
assert not (clips_v & clips_t),  "LEAK val↔test"
print(f"Train: {len(idx_tr)} mẫu / {len(clips_tr)} clip")
print(f"Val:   {len(idx_v)} mẫu / {len(clips_v)} clip")
print(f"Test:  {len(idx_t)} mẫu / {len(clips_t)} clip")

## 6. Train V1.2

In [ ]:
model = build_hybrid_model_56()
ckpt_path = f"{OUT_DIR}/best_v1.2_56feat_by_clip.keras"
cbs = [
    ModelCheckpoint(ckpt_path, monitor='val_loss', save_best_only=True),
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
]
t0 = time.time()
hist = model.fit(X_tr, y_tr, validation_data=(X_v, y_v),
                 epochs=100, batch_size=64, callbacks=cbs, verbose=2)
train_time = time.time() - t0
print(f"\nTraining time: {train_time:.1f}s, epochs trained: {len(hist.history['loss'])}")

## 7. Đánh giá trên test set

In [ ]:
y_pred = model.predict(X_t, verbose=0).argmax(1)
f1m = f1_score(y_t, y_pred, average='macro')
report = classification_report(y_t, y_pred, target_names=['Normal','Shoplifting'], digits=4)
cm = confusion_matrix(y_t, y_pred)
print(f"\n=== V1.2 — 56 geometric features + split-by-clip ===")
print(report)
print(f"Confusion matrix:\n{cm}")
print(f"F1-macro: {f1m:.4f}")

# Save log
log = {
    "config": "v1.2_56feat_by_clip",
    "input_shape": [90, 56],
    "train_size": int(X_tr.shape[0]),
    "val_size": int(X_v.shape[0]),
    "test_size": int(X_t.shape[0]),
    "epochs_trained": len(hist.history['loss']),
    "best_val_loss": float(min(hist.history['val_loss'])),
    "test_f1_macro": float(f1m),
    "test_confusion_matrix": cm.tolist(),
    "test_classification_report": report,
    "train_time_sec": train_time,
    "history": {k: [float(v) for v in vs] for k, vs in hist.history.items()},
}
with open(f"{OUT_DIR}/log_v1.2_56feat_by_clip.json", "w") as f:
    json.dump(log, f, indent=2, ensure_ascii=False)
print(f"\n✓ Saved {OUT_DIR}/log_v1.2_56feat_by_clip.json")

## 8. So sánh V1.1 (34 feat) vs V1.2 (56 feat)

In [ ]:
# Load lại log V1.1 nếu có
v11_log = None
try:
    with open(f"{OUT_DIR}/log_v1.1_split_by_clip.json") as f:
        v11_log = json.load(f)
    print(f"V1.1 F1: {v11_log['test_f1_macro']:.4f}")
except FileNotFoundError:
    print("Không thấy V1.1 log — chỉ in V1.2")
print(f"V1.2 F1: {f1m:.4f}")
if v11_log:
    delta = f1m - v11_log['test_f1_macro']
    print(f"\nΔ V1.2 - V1.1: {delta:+.4f} (cùng split-by-clip, khác chỉ feature)")
    if delta > 0.05:
        print("→ 56 geometric features CÓ giúp ích. Tiếp tục cải tiến.")
    elif delta > 0:
        print("→ 56 feat có cải thiện nhẹ. Có thể cần combine với regularization/aug.")
    else:
        print("→ 56 feat KHÔNG giúp. Vấn đề ở kiến trúc/dataset. Cần chuyển ST-GCN V2.0.")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist.history['accuracy'], label='train')
axes[0].plot(hist.history['val_accuracy'], label='val')
axes[0].set_title('V1.2 — Accuracy')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(hist.history['loss'], label='train')
axes[1].plot(hist.history['val_loss'], label='val')
axes[1].set_title('V1.2 — Loss')
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/v1.2_training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

# CM
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal','Shoplifting'], yticklabels=['Normal','Shoplifting'])
plt.title(f'V1.2 — 56 feat + by-clip\nF1={f1m:.4f}')
plt.xlabel('Dự đoán'); plt.ylabel('Thực tế')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/v1.2_cm.png", dpi=150, bbox_inches='tight')
plt.show()

## 9. Quyết định bước tiếp theo

| Kết quả V1.2 | Hành động |
|---|---|
| F1 ≥ 0,80 | Tuyệt vời — 56 feat đủ rồi. Tiếp tục V1.3 (regularize + augment) và V2.2 ensemble. |
| F1 ≈ 0,65-0,79 | Hứa hẹn — tiếp tục V1.3 + V2.0 ST-GCN để bứt phá lên ngưỡng tối thiểu. |
| F1 ≈ 0,55-0,65 | Yếu — vấn đề ở kiến trúc. Chuyển thẳng V2.0 ST-GCN. |
| F1 < 0,55 | Có thể bug — verify lại data processing. |

Gửi file `log_v1.2_56feat_by_clip.json` cho tôi để điền vào báo cáo Chương 3.5.1 (Bảng 3.2 mở rộng).